### Task X: Diffusion
Complete the specific task 2 from the DeepFalcon test. Comment on potential ideas to extend this classical diffusion architecture to a quantum diffusion and sketch out the architecture in detail.


In [1]:
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

The pt (transverse momentum) and mo (mass) keys often refer to the global properties of the entire jet, rather than the individual particles.

Tge data to use is X_jets and the labels are y.

In [2]:
with h5py.File("/content/quark-gluon_data-set_n139306.hdf5", "r") as f:
    print("Keys in file:", list(f.keys()))
    for key in f.keys():
        print(key, f[key].shape)


Keys in file: ['X_jets', 'm0', 'pt', 'y']
X_jets (139306, 125, 125, 3)
m0 (139306,)
pt (139306,)
y (139306,)


In [3]:
# 1. Dataset
class JetDataset(Dataset):
    def __init__(self, file_path, limit=None):
        self.file = h5py.File(file_path, 'r')
        self.points = self.file['X_jets']   # [N, 125, 125, 3]
        self.labels = self.file['y']
        if limit is not None:
            self.points = self.points[:limit]
            self.labels = self.labels[:limit]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Convert to PyTorch image format: (C, H, W)
        x = torch.tensor(self.points[idx], dtype=torch.float).permute(2, 0, 1)
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y

Diffusion CNN is the backbone network inside a diffusion model.

During training, the diffusion process:

- Takes a clean jet image (3 channels, 125×125).

- Adds Gaussian noise at a random timestep t (forward process).

- The model (DiffusionCNN) is then asked to predict the noise that was added.

In [4]:
# 2. DIFFUSION MODEL
class DiffusionCNN(nn.Module):
    def __init__(self, in_channels=3, hidden_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, hidden_dim, 3, padding=1)
        self.conv2 = nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1)
        self.conv3 = nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1)
        self.out = nn.Conv2d(hidden_dim, in_channels, 3, padding=1)

    def forward(self, x, t_emb):
        # Broadcast timestep embedding to image size
        t_emb = t_emb.view(-1, 1, 1, 1)
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))
        h = F.relu(self.conv3(h))
        return self.out(h)

That utility defines the forward diffusion step: it takes a clean image
𝑥
0
, adds Gaussian noise at a chosen timestep
𝑡
 using the pre‑computed noise schedule (
𝛽
,
𝛼
,
𝛼
cumprod
), and returns both the noisy image and the exact noise that was added.

In [5]:
# 3. DIFFUSION UTILITIES
def linear_beta_schedule(timesteps):
    return torch.linspace(1e-4, 0.02, timesteps)

timesteps = 50
betas = linear_beta_schedule(timesteps)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

def forward_diffusion_sample(x0, t):
    # Scalars per batch element
    sqrt_alphas_cumprod_t = alphas_cumprod[t].view(-1, 1, 1, 1) ** 0.5
    sqrt_one_minus_alphas_cumprod_t = (1 - alphas_cumprod[t]).view(-1, 1, 1, 1) ** 0.5
    noise = torch.randn_like(x0)
    return sqrt_alphas_cumprod_t * x0 + sqrt_one_minus_alphas_cumprod_t * noise, noise

In [6]:
# 4. TRAINING LOOP (SMALL SUBSET)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = JetDataset("/content/quark-gluon_data-set_n139306.hdf5", limit=200)  # 🔹 only 200 jets
loader = DataLoader(dataset, batch_size=8, shuffle=True)

model = DiffusionCNN(in_channels=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(15):  # 🔹 only 2 epochs for demo
    for batch_x, _ in loader:
        batch_x = batch_x.to(device)  # (B, 3, 125, 125)
        t = torch.randint(0, timesteps, (batch_x.size(0),), device=device)
        noisy, noise = forward_diffusion_sample(batch_x, t)
        pred_noise = model(noisy, t.float())
        loss = criterion(pred_noise, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss {loss.item():.4f}")


Epoch 1, Loss 0.4268
Epoch 2, Loss 0.0887
Epoch 3, Loss 0.2088
Epoch 4, Loss 0.0703
Epoch 5, Loss 0.1163
Epoch 6, Loss 0.0360
Epoch 7, Loss 0.0300
Epoch 8, Loss 0.1115
Epoch 9, Loss 0.0315
Epoch 10, Loss 0.0661
Epoch 11, Loss 0.0292
Epoch 12, Loss 0.0258
Epoch 13, Loss 0.0284
Epoch 14, Loss 0.1007
Epoch 15, Loss 0.0369


Since different batches are used in each epoch the loss goes up and down especially in the first few epochs but after that stabilizes well and the loss has a more clear decreasing trend as seen from the values above.